## Step 2: K-mer Analysis & Genome Size Estimation
**Input:** Trimmed/merged Illumina reads from `02-primary/merged/`  
**Output:** Genome size estimates across k-mer sizes 21–101, 
summary CSV and trend plots in `03-kmer-analysis/`  
**Tools:** KAT v2.4.2, R (ggplot2, dplyr, viridis)  
**Key parameters:** k-mer range: 21–101 (step 4 then 6 then 10); 
hash size: 250,000,000  
**Key finding:** Stable genome size estimate ~44 Mb; low heterozygosity  
**Reference:** Materials & Methods Section 4.1.4 — Nebli et al. (2025)

## Environment Variables

In [ ]:
export SN=3RR
export NCPUS=12

In [ ]:
alias kat="apptainer run docker://quay.io/biocontainers/kat:2.4.2--py36hc902310_3 kat"

In [ ]:
# Create main k-mer analysis directory
mkdir -p 03-kmer-analysis/{kat,plots,results}

# Create subdirectories for different analysis types
mkdir -p 03-kmer-analysis/kat/{option-A,option-B}

# K-mer Frequency Analysis

In [ ]:
export opt=A
for k in $(seq 21 4 39; seq 41 6 71 ; seq 71 10 101); do
    echo "Processing Option $opt (all files) with k-mer size: $k"
    kat hist \
        -o 03-kmer-analysis/kat/option-${opt}/hist-k${k} \
        -H 250000000 \
        -t $NCPUS \
        -m $k \
        -p pdf \
        02-primary/merged/${SN}-${opt}-illumina*.fastq.gz \
        > 03-kmer-analysis/kat/option-${opt}/hist-k${k}.out 2>&1
done


# Results Extraction and Summary

## Extract Genome Size Estimates

In [ ]:
# Create results summary file
echo "Option,KmerSize,GenomeSize" > 03-kmer-analysis/results/genome_size_estimates.csv

# Extract genome size estimates from Option A results
for opt in A; do
    for k in $(seq 21 4 39; seq 41 6 71 ; seq 71 10 101); do
        if [ -f "03-kmer-analysis/kat/option-${opt}/hist-k${k}.out" ]; then
            genome_size=$(grep "Estimated genome size" 03-kmer-analysis/kat/option-${opt}/hist-k${k}.out | \
                        awk '{print $4}' | sed 's/,//g')
            if [ ! -z "$genome_size" ]; then
                echo "${opt},${k},${genome_size}" >> 03-kmer-analysis/results/genome_size_estimates.csv
            fi
        fi
    done
done

# R configuration

## Create a personal library

In [ ]:
mkdir -p ~/Rlibs

## Install packages to personal library

In [ ]:
R -e 'install.packages(c("ggplot2","dplyr","readr","scales","viridis"), lib="~/Rlibs")'

## Load the packages in R

In [ ]:
nano visualize_kmer_analysis.R
# ----------------------------------------
# Load libraries from user library
# ----------------------------------------
.libPaths(c("~/Rlibs", .libPaths()))

library(ggplot2)
library(dplyr)
library(readr)
library(scales)
library(viridis)

# ----------------------------------------
# Define base directory
# ----------------------------------------
base_dir <- "/home/fbouzid/SN"  # <-- change this to your absolute project path

# Paths for input and output
input_file <- file.path(base_dir, "03-kmer-analysis", "results", "genome_size_estimates.csv")
plot_dir <- file.path(base_dir, "03-kmer-analysis", "plots")

# Create plots directory if it doesn't exist
if (!dir.exists(plot_dir)) {
  dir.create(plot_dir, recursive = TRUE)
}

# ----------------------------------------
# Load and preprocess data
# ----------------------------------------
data <- read_csv(input_file)

# Convert GenomeSize to numeric (handle potential formatting issues)
data$GenomeSize <- as.numeric(gsub(",", "", data$GenomeSize))

# Create factor levels for proper ordering
data$Option <- factor(data$Option, levels = c("A", "B"))

# Remove aberrant values
data <- filter(data, .data$GenomeSize < 80)

# ----------------------------------------
# Create detailed comparison plot
# ----------------------------------------
p1 <- ggplot(data, aes(x = KmerSize, y = GenomeSize, color = Option)) +
  geom_point(size = 2.5, alpha = 0.8) +
  geom_line(aes(group = Option), size = 1, alpha = 0.7) +
  geom_smooth(aes(group = Option), method = "loess", se = TRUE, alpha = 0.2) +
  scale_color_viridis_d(name = "Preprocessing\nOption") +
  scale_x_continuous(breaks = c(5, 15, 25, 35, 61, 81, 91, 101, 111, 121, 131)) +
  scale_y_continuous(labels = comma_format()) +
  labs(title = "Genome Size Estimates with Trend Lines",
       subtitle = "Smoothed trends showing convergence patterns",
       x = "K-mer Size",
       y = "Estimated Genome Size (Mb)") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(face = "bold"))

# ----------------------------------------
# Save plots
# ----------------------------------------
ggsave(file.path(plot_dir, "genome_size_trends.png"), p1, width = 12, height = 6, dpi = 300)
ggsave(file.path(plot_dir, "genome_size_trends.pdf"), p1, width = 12, height = 6)


In [ ]:
Rscript visualize_kmer_analysis.R